# Task 1: Symbolic, Unconditioned Generation
##### Dataset: EMOPIA_1.0 
##### Output file: symbolic_unconditioned_mid 

## Setup


In [1]:
# EMOPIA_1.0 folder should be in the same folder as this notebook before running.

from __future__ import annotations

import csv
import html
import math
import random
import re
import statistics
import struct
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path


DATASET_DIR = Path("EMOPIA_1.0")
OUTPUT_MIDI = Path("symbolic_unconditioned.mid")
OUTPUT_HTML = Path("workbook.html")
RANDOM_SEED = 153253

TIME_SHIFT_PREFIX = "TS_"
NOTE_PREFIX = "N_"
START = "<START>"
END = "<END>"

## Data Containers 
The following dataclasses store parsed MIDI notes and parsed MIDI clips. 

In [2]:
@dataclass
class Note:
    start: int
    end: int
    pitch: int
    velocity: int

@dataclass
class MidiClip:
    path: Path
    ticks_per_beat: int
    notes: list[Note]

## MIDI low-level helpers 
MIDI files store many time values as variable-length quantities, the following helper functions read and wirte that compact format. 

In [3]:
def read_vlq(data: bytes, pos: int) -> tuple[int, int]:
    value = 0
    while True:
        if pos >= len(data):
            raise ValueError("Unexpected end of data while reading variable-length integer")
        byte = data[pos]
        pos += 1
        value = (value << 7) | (byte & 0x7F)
        if not byte & 0x80:
            return value, pos


def write_vlq(value: int) -> bytes:
    if value < 0:
        raise ValueError("VLQ values must be non-negative")
    buffer = value & 0x7F
    value >>= 7
    while value:
        buffer <<= 8
        buffer |= (value & 0x7F) | 0x80
        value >>= 7
    out = bytearray()
    while True:
        out.append(buffer & 0xFF)
        if buffer & 0x80:
            buffer >>= 8
        else:
            break
    return bytes(out)


## MIDI Parsing
The following functions will cover functionality revolving around reading each MIDI file, walking though track events, pairing note-on/note-off events, and return a list of notes.

In [4]:
def parse_midi(path: Path) -> MidiClip:
    data = path.read_bytes()
    pos = 0
    if data[pos : pos + 4] != b"MThd":
        raise ValueError(f"{path} is not a MIDI file")
    pos += 4
    header_length = struct.unpack(">I", data[pos : pos + 4])[0]
    pos += 4
    fmt, num_tracks, division = struct.unpack(">HHH", data[pos : pos + 6])
    pos += header_length
    if division & 0x8000:
        raise ValueError(f"{path} uses SMPTE time division, which this script does not support")

    all_notes: list[Note] = []
    for _ in range(num_tracks):
        if data[pos : pos + 4] != b"MTrk":
            raise ValueError(f"{path} has malformed track header")
        pos += 4
        track_length = struct.unpack(">I", data[pos : pos + 4])[0]
        pos += 4
        track_data = data[pos : pos + track_length]
        pos += track_length
        all_notes.extend(parse_track_notes(track_data))

    all_notes.sort(key=lambda n: (n.start, n.pitch, n.end))
    return MidiClip(path=path, ticks_per_beat=division, notes=all_notes)


def parse_track_notes(track: bytes) -> list[Note]:
    notes: list[Note] = []
    active: dict[tuple[int, int], list[tuple[int, int]]] = defaultdict(list)
    tick = 0
    pos = 0
    running_status: int | None = None

    while pos < len(track):
        delta, pos = read_vlq(track, pos)
        tick += delta
        if pos >= len(track):
            break

        status_or_data = track[pos]
        if status_or_data & 0x80:
            status = status_or_data
            pos += 1
            if status < 0xF0:
                running_status = status
        else:
            if running_status is None:
                raise ValueError("MIDI running status used before status byte")
            status = running_status

        if status == 0xFF:
            if pos >= len(track):
                break
            meta_type = track[pos]
            pos += 1
            length, pos = read_vlq(track, pos)
            pos += length
            if meta_type == 0x2F:
                break
            continue

        if status in (0xF0, 0xF7):
            length, pos = read_vlq(track, pos)
            pos += length
            continue

        event_type = status & 0xF0
        channel = status & 0x0F
        data_len = 1 if event_type in (0xC0, 0xD0) else 2
        event_data = track[pos : pos + data_len]
        pos += data_len
        if len(event_data) < data_len:
            break

        if event_type == 0x90:
            pitch, velocity = event_data[0], event_data[1]
            key = (channel, pitch)
            if velocity > 0:
                active[key].append((tick, velocity))
            elif active[key]:
                start, start_velocity = active[key].pop(0)
                if tick > start:
                    notes.append(Note(start, tick, pitch, start_velocity))
        elif event_type == 0x80:
            pitch = event_data[0]
            key = (channel, pitch)
            if active[key]:
                start, start_velocity = active[key].pop(0)
                if tick > start:
                    notes.append(Note(start, tick, pitch, start_velocity))

    return notes

## Tokenization 
This section is responsible for converting note events into symbolic language-model tokens. Time-shift tokens advance time, and note tokens add pitches at the current time. 

In [5]:
def quantize_notes(clip: MidiClip, max_duration_steps: int = 32) -> list[tuple[int, int, int, int]]:
    step_ticks = max(1, round(clip.ticks_per_beat / 4))
    quantized = []
    for note in clip.notes:
        start_step = max(0, round(note.start / step_ticks))
        duration_steps = max(1, round((note.end - note.start) / step_ticks))
        duration_steps = min(max_duration_steps, duration_steps)
        velocity_bin = min(7, max(1, round(note.velocity / 16)))
        if 0 <= note.pitch <= 127:
            quantized.append((start_step, note.pitch, duration_steps, velocity_bin))
    quantized.sort(key=lambda n: (n[0], n[1], n[2], n[3]))
    return quantized


def clip_to_tokens(clip: MidiClip) -> list[str]:
    tokens = [START]
    current_step = 0
    for start_step, pitch, duration_steps, velocity_bin in quantize_notes(clip):
        gap = start_step - current_step
        while gap > 0:
            shift = min(16, gap)
            tokens.append(f"{TIME_SHIFT_PREFIX}{shift}")
            current_step += shift
            gap -= shift
        tokens.append(f"{NOTE_PREFIX}{pitch}_{duration_steps}_{velocity_bin}")
    tokens.append(END)
    return tokens


def parse_note_token(token: str) -> tuple[int, int, int] | None:
    if not token.startswith(NOTE_PREFIX):
        return None
    parts = token[len(NOTE_PREFIX) :].split("_")
    if len(parts) != 3:
        return None
    try:
        pitch, duration, velocity_bin = (int(part) for part in parts)
    except ValueError:
        return None
    if not 0 <= pitch <= 127:
        return None
    return pitch, max(1, duration), min(7, max(1, velocity_bin))


def token_time_shift(token: str) -> int:
    if token.startswith(TIME_SHIFT_PREFIX):
        try:
            return max(0, int(token[len(TIME_SHIFT_PREFIX) :]))
        except ValueError:
            return 0
    return 0


## N-gram Model
This is the generative model. It counts token contexts, computes next-token probabilities with backoff, and samples new token sequences. 

In [6]:
class NGramModel:
    def __init__(self, order: int = 4, alpha: float = 0.8, seed: int = RANDOM_SEED):
        self.order = order
        self.alpha = alpha
        self.rng = random.Random(seed)
        self.counts: list[defaultdict[tuple[str, ...], Counter[str]]] = [
            defaultdict(Counter) for _ in range(order + 1)
        ]
        self.vocab_counter: Counter[str] = Counter()
        self.vocab: list[str] = []

    def fit(self, sequences: list[list[str]]) -> None:
        for sequence in sequences:
            self.vocab_counter.update(sequence)
            for index, token in enumerate(sequence):
                max_context = min(self.order - 1, index)
                for context_size in range(max_context + 1):
                    context = tuple(sequence[index - context_size : index])
                    self.counts[context_size + 1][context][token] += 1
        self.vocab = sorted(self.vocab_counter)

    def probability(self, context: tuple[str, ...], token: str) -> float:
        context = context[-(self.order - 1) :]
        return self._prob_recursive(context, token)

    def _prob_recursive(self, context: tuple[str, ...], token: str) -> float:
        vocab_size = max(1, len(self.vocab))
        if not context:
            counter = self.counts[1].get((), Counter())
            total = sum(counter.values())
            return (counter[token] + self.alpha) / (total + self.alpha * vocab_size)

        counter = self.counts[len(context) + 1].get(context)
        if not counter:
            return self._prob_recursive(context[1:], token)
        total = sum(counter.values())
        unique = len(counter)
        mle = counter[token] / total if total else 0.0
        backoff = self._prob_recursive(context[1:], token)
        weight = total / (total + self.alpha * unique)
        return weight * mle + (1.0 - weight) * backoff

    def sample_next(
        self,
        context: tuple[str, ...],
        temperature: float = 1.05,
        banned: set[str] | None = None,
    ) -> str:
        banned = banned or set()
        context = context[-(self.order - 1) :]
        counter = None
        while context:
            counter = self.counts[len(context) + 1].get(context)
            if counter:
                break
            context = context[1:]
        if not counter:
            counter = self.counts[1][()]

        candidates = [(token, count) for token, count in counter.items() if token not in banned]
        if not candidates:
            candidates = [(token, count) for token, count in self.vocab_counter.items() if token not in banned]
        if not candidates:
            return END

        inv_temp = 1.0 / max(0.05, temperature)
        weights = [max(count, 1) ** inv_temp for _, count in candidates]
        total = sum(weights)
        pick = self.rng.random() * total
        running = 0.0
        for (token, _), weight in zip(candidates, weights):
            running += weight
            if running >= pick:
                return token
        return candidates[-1][0]

    def generate(
        self,
        target_steps: int = 512,
        max_tokens: int = 5000,
        temperature: float = 1.05,
    ) -> list[str]:
        sequence = [START]
        steps = 0
        note_count = 0
        for _ in range(max_tokens):
            banned = {START}
            if steps < target_steps * 0.75:
                banned.add(END)
            token = self.sample_next(tuple(sequence), temperature=temperature, banned=banned)
            if token == END and steps >= target_steps * 0.75:
                sequence.append(token)
                break
            shift = token_time_shift(token)
            parsed_note = parse_note_token(token)
            if shift:
                steps += shift
                sequence.append(token)
            elif parsed_note:
                sequence.append(token)
                note_count += 1
            elif token != START:
                sequence.append(token)
            if steps >= target_steps and note_count >= 128:
                sequence.append(END)
                break
        if sequence[-1] != END:
            sequence.append(END)
        return sequence


## Evaluation & MIDI Output 
These utilities compte perplexity, summarize generated music, covnert sampled tokens back to notes, and write a new MIDI file.

In [7]:
def perplexity(model: NGramModel, sequences: list[list[str]]) -> float:
    total_log = 0.0
    total_tokens = 0
    for sequence in sequences:
        for index in range(1, len(sequence)):
            context = tuple(sequence[max(0, index - model.order + 1) : index])
            prob = max(model.probability(context, sequence[index]), 1e-12)
            total_log += math.log(prob)
            total_tokens += 1
    return math.exp(-total_log / max(1, total_tokens))


def sequence_metrics(sequence: list[str]) -> dict[str, float]:
    pitches = []
    durations = []
    velocity_bins = []
    time_steps = 0
    note_count = 0
    for token in sequence:
        time_steps += token_time_shift(token)
        parsed = parse_note_token(token)
        if parsed:
            pitch, duration, velocity_bin = parsed
            pitches.append(pitch)
            durations.append(duration)
            velocity_bins.append(velocity_bin)
            note_count += 1
    unique_pitches = len(set(pitches))
    pitch_range = max(pitches) - min(pitches) if pitches else 0
    avg_pitch = statistics.fmean(pitches) if pitches else 0.0
    avg_duration = statistics.fmean(durations) if durations else 0.0
    avg_velocity = statistics.fmean(velocity_bins) if velocity_bins else 0.0
    density = note_count / max(1, time_steps / 16)
    return {
        "notes": note_count,
        "steps": time_steps,
        "unique_pitches": unique_pitches,
        "pitch_range": pitch_range,
        "avg_pitch": avg_pitch,
        "avg_duration": avg_duration,
        "avg_velocity_bin": avg_velocity,
        "notes_per_bar": density,
    }


def tokens_to_notes(sequence: list[str], ticks_per_beat: int = 480) -> list[Note]:
    step_ticks = ticks_per_beat // 4
    current_step = 0
    notes = []
    for token in sequence:
        shift = token_time_shift(token)
        if shift:
            current_step += shift
            continue
        parsed = parse_note_token(token)
        if not parsed:
            continue
        pitch, duration_steps, velocity_bin = parsed
        velocity = min(85, max(20, velocity_bin * 10))
        start = current_step * step_ticks
        end = (current_step + max(duration_steps * 2, 4)) * step_ticks
        notes.append(Note(start=start, end=end, pitch=pitch, velocity=velocity))
    return notes


def write_midi(path: Path, notes: list[Note], ticks_per_beat: int = 480, bpm: int = 96, program: int = 48) -> None:
    events: list[tuple[int, int, bytes]] = []
    tempo = round(60_000_000 / bpm)
    events.append((0, 0, b"\xFF\x51\x03" + tempo.to_bytes(3, "big")))
    program = min(127, max(0, program))
    events.append((0, 1, bytes([0xC0, program])))
    for note in notes:
        pitch = min(127, max(0, note.pitch))
        velocity = min(127, max(1, note.velocity))
        events.append((note.start, 2, bytes([0x90, pitch, velocity])))
        events.append((note.end, 1, bytes([0x80, pitch, 0])))
    events.sort(key=lambda event: (event[0], event[1]))

    track = bytearray()
    last_tick = 0
    for tick, _, payload in events:
        track += write_vlq(max(0, tick - last_tick))
        track += payload
        last_tick = tick
    track += write_vlq(ticks_per_beat)
    track += b"\xFF\x2F\x00"

    header = b"MThd" + struct.pack(">IHHH", 6, 0, 1, ticks_per_beat)
    track_chunk = b"MTrk" + struct.pack(">I", len(track)) + bytes(track)
    path.write_bytes(header + track_chunk)


## Dataset Loading & Train/Val/Test Split 
These functions read EMOPIA labels, group clips by source song, split by song ID to avoid leakage, and parse all usable MIDI files. 

In [8]:
def load_labels(dataset_dir: Path) -> dict[str, dict[str, str]]:
    label_path = dataset_dir / "label.csv"
    with label_path.open(newline="") as f:
        rows = list(csv.DictReader(f))
    return {row["ID"]: row for row in rows}


def song_id_from_clip_id(clip_id: str) -> str:
    without_q = re.sub(r"^Q[1-4]_", "", clip_id)
    return without_q.rsplit("_", 1)[0]


def split_by_song(labels: dict[str, dict[str, str]]) -> dict[str, str]:
    rng = random.Random(RANDOM_SEED)
    song_ids = sorted({song_id_from_clip_id(clip_id) for clip_id in labels})
    rng.shuffle(song_ids)
    n = len(song_ids)
    train_cut = round(n * 0.8)
    val_cut = round(n * 0.9)
    split_map = {}
    for song_id in song_ids[:train_cut]:
        split_map[song_id] = "train"
    for song_id in song_ids[train_cut:val_cut]:
        split_map[song_id] = "val"
    for song_id in song_ids[val_cut:]:
        split_map[song_id] = "test"
    return split_map


def load_sequences(dataset_dir: Path) -> tuple[list[dict[str, object]], dict[str, dict[str, str]]]:
    labels = load_labels(dataset_dir)
    split_map = split_by_song(labels)
    records = []
    for midi_path in sorted((dataset_dir / "midis").glob("*.mid")):
        clip_id = midi_path.stem
        if clip_id not in labels:
            continue
        try:
            clip = parse_midi(midi_path)
            if not clip.notes:
                continue
            tokens = clip_to_tokens(clip)
        except Exception as exc:
            print(f"Skipping {midi_path}: {exc}")
            continue
        song_id = song_id_from_clip_id(clip_id)
        records.append(
            {
                "clip_id": clip_id,
                "song_id": song_id,
                "quadrant": labels[clip_id]["4Q"],
                "annotator": labels[clip_id]["annotator"],
                "split": split_map[song_id],
                "tokens": tokens,
                "notes": len(clip.notes),
                "ticks_per_beat": clip.ticks_per_beat,
            }
        )
    return records, labels


## Report Helper Functions 
Small helpers for averages, counts, HTML tables, SVG bars, and dataset summary statistics. 

In [9]:
def mean(values: list[float]) -> float:
    return statistics.fmean(values) if values else 0.0


def pct(value: float) -> str:
    return f"{100 * value:.1f}%"


def count_by(records: list[dict[str, object]], key: str) -> Counter[str]:
    return Counter(str(record[key]) for record in records)


def html_table(headers: list[str], rows: list[list[object]]) -> str:
    out = ["<table>", "<thead><tr>"]
    out.extend(f"<th>{html.escape(str(header))}</th>" for header in headers)
    out.append("</tr></thead><tbody>")
    for row in rows:
        out.append("<tr>")
        out.extend(f"<td>{html.escape(str(value))}</td>" for value in row)
        out.append("</tr>")
    out.append("</tbody></table>")
    return "\n".join(out)


def bar_svg(counter: Counter[str], title: str) -> str:
    width = 680
    bar_height = 28
    gap = 10
    left = 120
    max_count = max(counter.values()) if counter else 1
    height = 42 + len(counter) * (bar_height + gap)
    rows = [
        f'<svg viewBox="0 0 {width} {height}" role="img" aria-label="{html.escape(title)}">',
        f'<text x="0" y="18" class="chart-title">{html.escape(title)}</text>',
    ]
    y = 34
    for label, count in sorted(counter.items()):
        bar_width = round((width - left - 20) * count / max_count)
        rows.append(f'<text x="0" y="{y + 19}" class="chart-label">{html.escape(label)}</text>')
        rows.append(f'<rect x="{left}" y="{y}" width="{bar_width}" height="{bar_height}" rx="3"></rect>')
        rows.append(f'<text x="{left + bar_width + 8}" y="{y + 19}" class="chart-value">{count}</text>')
        y += bar_height + gap
    rows.append("</svg>")
    return "\n".join(rows)


def describe_records(records: list[dict[str, object]]) -> dict[str, object]:
    note_counts = [float(record["notes"]) for record in records]
    token_counts = [float(len(record["tokens"])) for record in records]
    song_count = len({record["song_id"] for record in records})
    return {
        "clips": len(records),
        "songs": song_count,
        "mean_notes": mean(note_counts),
        "mean_tokens": mean(token_counts),
        "total_notes": int(sum(note_counts)),
        "total_tokens": int(sum(token_counts)),
        "quadrants": count_by(records, "quadrant"),
        "splits": count_by(records, "split"),
    }


## HTML Report Builder 
This section builds the final "workbook.html" file with dataset summary, modeling description, evaluations, and related work. This is to provide whomever is interested with a clean web page report they can access throuhg their respective browser. 

In [10]:
def build_report(
    records: list[dict[str, object]],
    generated_tokens: list[str],
    eval_rows: list[list[object]],
    model_order: int,
    output_midi: Path,
    generation_temperature: float,
    target_steps: int,
    bpm: int,
    midi_program: int,
    sound_preset: str,
) -> str:
    summary = describe_records(records)
    generated = sequence_metrics(generated_tokens)
    train_records = [record for record in records if record["split"] == "train"]
    test_records = [record for record in records if record["split"] == "test"]
    train_metrics = [sequence_metrics(record["tokens"]) for record in train_records]
    test_metrics = [sequence_metrics(record["tokens"]) for record in test_records]

    metric_rows = []
    for metric, label in [
        ("notes", "Notes per clip"),
        ("unique_pitches", "Unique pitches"),
        ("pitch_range", "Pitch range"),
        ("avg_duration", "Mean duration (16th-note steps)"),
        ("notes_per_bar", "Notes per 4/4 bar"),
    ]:
        metric_rows.append(
            [
                label,
                f"{mean([m[metric] for m in train_metrics]):.2f}",
                f"{mean([m[metric] for m in test_metrics]):.2f}",
                f"{generated[metric]:.2f}",
            ]
        )

    split_rows = [[split, count] for split, count in sorted(summary["splits"].items())]
    quadrant_rows = [[f"Q{q}", count] for q, count in sorted(summary["quadrants"].items())]
    generated_rows = [[key.replace("_", " "), f"{value:.2f}"] for key, value in generated.items()]
    generation_rows = [
        ["Selected model", f"{model_order}-gram"],
        ["Target length", f"{target_steps} sixteenth-note steps, approximately {target_steps / 16:.1f} bars"],
        ["Sampling temperature", f"{generation_temperature:.2f}"],
        ["Tempo", f"{bpm} BPM"],
        ["MIDI program", f"{midi_program}"],
        ["Sound/post-processing", sound_preset],
    ]

    return f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<title>CSE 153/253 Assignment 2 - Task 1</title>
<style>
  :root {{
    --ink: #202124;
    --muted: #5f6368;
    --line: #d9dee5;
    --paper: #ffffff;
    --soft: #f6f8fa;
    --accent: #2f6f6f;
    --accent-2: #8c4b2f;
  }}
  body {{
    margin: 0;
    background: var(--paper);
    color: var(--ink);
    font: 16px/1.55 -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
  }}
  main {{
    max-width: 980px;
    margin: 0 auto;
    padding: 38px 24px 72px;
  }}
  h1, h2, h3 {{
    line-height: 1.2;
    letter-spacing: 0;
  }}
  h1 {{
    font-size: 34px;
    margin: 0 0 8px;
  }}
  h2 {{
    border-top: 1px solid var(--line);
    margin-top: 34px;
    padding-top: 24px;
    font-size: 24px;
  }}
  h3 {{
    margin-top: 22px;
    font-size: 18px;
  }}
  p {{
    margin: 10px 0;
  }}
  .lede {{
    color: var(--muted);
    font-size: 18px;
  }}
  code {{
    background: var(--soft);
    padding: 0 4px;
    border-radius: 4px;
  }}
  table {{
    border-collapse: collapse;
    width: 100%;
    margin: 14px 0 24px;
    font-size: 14px;
  }}
  th, td {{
    border: 1px solid var(--line);
    padding: 8px 10px;
    text-align: left;
    vertical-align: top;
  }}
  th {{
    background: var(--soft);
  }}
  svg {{
    display: block;
    width: 100%;
    max-width: 720px;
    margin: 12px 0 20px;
  }}
  rect {{
    fill: var(--accent);
  }}
  .chart-title {{
    font-weight: 700;
    fill: var(--ink);
    font-size: 16px;
  }}
  .chart-label, .chart-value {{
    fill: var(--muted);
    font-size: 13px;
  }}
  .callout {{
    background: #f8f4ef;
    border-left: 4px solid var(--accent-2);
    padding: 12px 14px;
    margin: 18px 0;
  }}
</style>
</head>
<body>
<main>
<h1>Task 1: Symbolic, Unconditioned Generation</h1>
<p class="lede">Dataset: <code>EMOPIA_1.0</code>. Output file: <code>{html.escape(str(output_midi))}</code>.</p>

<h2>1. Exploratory Analysis and Preprocessing</h2>
<p>
EMOPIA is a pop-piano dataset organized around perceived emotion. The local copy
contains transcribed MIDI clips, YouTube metadata, raw tagging lists, and a
<code>label.csv</code> file with one of four valence-arousal quadrants for each
clip. This task ignores the emotion labels during generation because it is
unconditioned, but the labels are still useful for checking coverage and making
a song-level split.
</p>
<p>
I parsed the MIDI files directly, extracted note-on/note-off pairs, quantized
time to sixteenth-note steps, and represented each clip as an event sequence:
<code>TS_k</code> advances time by <code>k</code> sixteenth-note steps and
<code>N_pitch_duration_velocity</code> places a note at the current time.
Durations are capped at 32 steps and velocities are bucketed into 7 nonzero
bins. Splits are made by YouTube song ID so clips from the same source song do
not appear in both training and test sets.
</p>
<p>
After generation, I also applied a small sound-design post-processing step to
make the MIDI less percussive: generated note velocities are softened, very
short notes are given a longer minimum duration, the tempo is lowered, and the
instrument is changed from piano to a string ensemble. These changes affect the
rendered sound but not the symbolic n-gram training objective.
</p>
{html_table(["Statistic", "Value"], [
    ["MIDI clips parsed", summary["clips"]],
    ["Unique source songs", summary["songs"]],
    ["Total extracted notes", summary["total_notes"]],
    ["Total event tokens", summary["total_tokens"]],
    ["Mean notes per clip", f"{summary["mean_notes"]:.2f}"],
    ["Mean tokens per clip", f"{summary["mean_tokens"]:.2f}"],
])}
{html_table(["Split", "Clips"], split_rows)}
{bar_svg(summary["quadrants"], "Clip counts by EMOPIA quadrant")}
{html_table(["Quadrant", "Clips"], quadrant_rows)}

<h2>2. Modeling</h2>
<p>
The ML problem is next-event modeling: estimate
<code>p(x_t | x_{{t-1}}, ..., x_{{t-n+1}})</code> over symbolic MIDI events,
then sample events autoregressively. I trained a variable-order n-gram model
with maximum order {model_order}. When an exact context is rare or unseen, the
model interpolates with shorter contexts, ending in a smoothed unigram model.
This is more expressive than a first-order Markov chain because it can preserve
short timing and note-pattern motifs, but it remains transparent and fast enough
to train from scratch on the full dataset without GPUs or external packages.
</p>
<p>
Compared with an RNN or Transformer, this model has limited long-range form: it
does not explicitly know about phrase boundaries, harmonic goals, or repetition
over many measures. Its advantages are that it is robust on a small dataset,
easy to audit, and directly optimizes a symbolic likelihood aligned with the
generation process.
</p>
<p>
For the final sample, I increased the target length to make the generated piece
longer and lowered the sampling temperature below 1.0. Lower temperature makes
the model choose more likely next events, which usually produces safer, smoother,
and less chaotic output.
</p>
{html_table(["Generation setting", "Value"], generation_rows)}

<h2>3. Evaluation</h2>
<p>
For objective evaluation, I compare held-out next-token perplexity for unigram,
bigram, trigram, and four-gram models. Lower perplexity means the model assigns
higher probability to unseen EMOPIA clips. This does not prove musical quality:
a model can have good perplexity but still wander harmonically. So I also report
simple musical descriptors of the generated sample beside train/test averages.
</p>
{html_table(["Model", "Validation perplexity", "Test perplexity"], eval_rows)}
{html_table(["Descriptor", "Train mean", "Test mean", "Generated sample"], metric_rows)}
<div class="callout">
The generated sample uses the validation-selected {model_order}-gram model with
temperature {generation_temperature:.2f}, target length {target_steps} steps, and tempo {bpm} BPM.
The lower temperature reduces randomness, while the longer target length gives a
longer song. The string ensemble program, softer velocities, and longer minimum
note durations reduce the tapping effect caused by frequent short note attacks.
</div>
{html_table(["Generated descriptor", "Value"], generated_rows)}

<h2>4. Related Work</h2>
<p>
The EMOPIA paper introduced the dataset for emotion recognition and
emotion-based music generation in pop piano. It pairs symbolic MIDI with
emotion annotations in the four-quadrant valence-arousal space, which makes it
natural for conditioned generation. This Task 1 model deliberately leaves the
emotion labels out to estimate the overall symbolic distribution instead.
</p>
<p>
Prior symbolic music generation systems often use event-token language models:
older systems use Markov or n-gram variants, while recent systems use recurrent
networks or Transformer-like decoders. The same evaluation tension appears
throughout this literature: likelihood-style metrics are useful for debugging,
but final musical quality usually needs listening tests or human judgment.
</p>
<p>
References: Hung et al., <a href="https://arxiv.org/abs/2108.01374">EMOPIA: A
Multi-Modal Pop Piano Dataset for Emotion Recognition and Emotion-based Music
Generation</a>; project page at <a href="https://annahung31.github.io/EMOPIA/">annahung31.github.io/EMOPIA</a>.
</p>
</main>
</body>
</html>
"""



## Run the Full Pipeline 
This is the main pipeline for this task, which is responsible for the followoing: load data, train 1-4 gram models, pick the best validation model, generate MIDI, and write the HTML report. 

In [11]:
def main() -> None:
    if not DATASET_DIR.exists():
        raise SystemExit(f"Dataset directory not found: {DATASET_DIR}")

    print("Loading and tokenizing EMOPIA MIDI clips...")
    records, _ = load_sequences(DATASET_DIR)
    if not records:
        raise SystemExit("No usable MIDI records found")

    sequences_by_split = {
        split: [record["tokens"] for record in records if record["split"] == split]
        for split in ("train", "val", "test")
    }

    print(
        "Parsed "
        f"{len(records)} clips from {len({record['song_id'] for record in records})} songs "
        f"({len(sequences_by_split['train'])} train, "
        f"{len(sequences_by_split['val'])} val, {len(sequences_by_split['test'])} test)."
    )

    eval_rows = []
    validation_scores = {}
    models = {}
    for order in (1, 2, 3, 4):
        model = NGramModel(order=order, seed=RANDOM_SEED + order)
        model.fit(sequences_by_split["train"])
        models[order] = model
        val_ppl = perplexity(model, sequences_by_split["val"])
        test_ppl = perplexity(model, sequences_by_split["test"])
        validation_scores[order] = val_ppl
        eval_rows.append([f"{order}-gram", f"{val_ppl:.2f}", f"{test_ppl:.2f}"])
        print(f"{order}-gram perplexity: val={val_ppl:.2f}, test={test_ppl:.2f}")

    best_order = min(validation_scores, key=validation_scores.get)
    print(f"Selected {best_order}-gram for generation by validation perplexity.")

    # Final generation settings.
    # These settings were chosen to make the generated MIDI longer and smoother:
    # - target_steps=1024 makes the piece about twice as long as the original 512-step version.
    # - temperature=0.85 makes sampling less random.
    # - bpm=72 slows down the note attacks.
    # - midi_program=48 switches from piano to string ensemble.
    target_steps = 1024
    generation_temperature = 0.85
    bpm = 72
    midi_program = 48
    sound_preset = "String ensemble; softened velocity; doubled/minimum note duration"

    generated_tokens = models[best_order].generate(
        target_steps=target_steps,
        temperature=generation_temperature,
    )
    generated_notes = tokens_to_notes(generated_tokens)
    write_midi(OUTPUT_MIDI, generated_notes, bpm=bpm, program=midi_program)
    print(f"Wrote {OUTPUT_MIDI} with {len(generated_notes)} notes.")

    html_report = build_report(
        records=records,
        generated_tokens=generated_tokens,
        eval_rows=eval_rows,
        model_order=best_order,
        output_midi=OUTPUT_MIDI,
        generation_temperature=generation_temperature,
        target_steps=target_steps,
        bpm=bpm,
        midi_program=midi_program,
        sound_preset=sound_preset,
    )
    OUTPUT_HTML.write_text(html_report, encoding="utf-8")
    print(f"Wrote {OUTPUT_HTML}.")

## Execute the pipeline 
Run this cell last to run the full experiment and output symbolic_unconditioned_mid

In [12]:
# Uncomment this when you want to run
main()

Loading and tokenizing EMOPIA MIDI clips...
Parsed 1078 clips from 387 songs (887 train, 89 val, 102 test).
1-gram perplexity: val=818.22, test=775.07
2-gram perplexity: val=602.00, test=590.14
3-gram perplexity: val=863.99, test=831.25
4-gram perplexity: val=980.90, test=932.55
Selected 2-gram for generation by validation perplexity.
Wrote symbolic_unconditioned.mid with 923 notes.
Wrote workbook.html.
